# Anima / SDXL 生图 + Wan 视频 - Colab 免费档实测版

面向**免费档 T4 16GB / 12.6GB 系统内存 / 102GB 临时盘 / 3.5-6h 会话**。

## 档位自适应

| 档位 | 能做什么 | 本 notebook 行为 |
|---|---|---|
| **GPU 档**（T4 16GB） | Anima 生图 5-10s、SDXL 15-40s、Wan 2.2 5B 视频 | 全功能 |
| **CPU 档**（无卡） | 只跑 ffmpeg 后处理 / 打标 / 编解码 | 自动降级，禁用生成，服务仍可用 |

## 修复与新增（相对旧版）

- **fp16 修复**：T4 是 Turing，没有 bf16，`bfloat16` 会掉到无意义的软件模拟。已按卡自动选 `fp16`。
- **会话保活**：免费档约 90 分钟无交互断线，新增 JS 保活。
- **产出落盘**：`/content` 是临时盘（会话结束即清空），新增自动上传 Drive。
- **批量模式**：一次提交 N 条 prompt，整批出图后统一下载。
- **视频支持**：Wan 2.2 TI2V-5B（8GB 级）图生视频，含 ffmpeg 合流。
- **时长纠正**：Colab 免费档**不是 12 小时**，实测 3.5-6h，按 3h 规划。

跑完第 4 格打印 `https://xxx.trycloudflare.com`，本机用 `colab_gen.sh "prompt" <url>` 取图。


In [ ]:
#@title 1. 环境探测 + 依赖安装（先跑这格）
import os, sys, subprocess, shutil

IN_COLAB = os.path.isdir('/content')
print('colab:', IN_COLAB)

# 内存：12.6GB 是免费档标准系统内存，不是异常
try:
    import psutil
    vm = psutil.virtual_memory()
    print(f'RAM : {vm.total/1024**3:.1f} GB total, {vm.available/1024**3:.1f} GB free')
except ImportError:
    vm = None

du = shutil.disk_usage('/content' if IN_COLAB else '.')
print(f'disk: {du.free/1024**3:.1f} GB free')

# GPU 探测：决定走 GPU 还是 CPU 路径
HAS_GPU = False
GPU_NAME, VRAM_GB = 'none', 0.0
try:
    out = subprocess.run(
        ['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
        capture_output=True, text=True, timeout=30)
    if out.returncode == 0 and out.stdout.strip():
        line = out.stdout.strip().split('\n')[0]
        GPU_NAME = line.split(',')[0].strip()
        VRAM_GB = float(line.split(',')[1].replace('MiB', '').strip()) / 1024
        HAS_GPU = True
except Exception as e:
    print('nvidia-smi failed:', e)

print(f'GPU : {GPU_NAME} ({VRAM_GB:.1f} GB)' if HAS_GPU else 'GPU : 无 -> CPU 档，只能做后处理')

# fp16 for everything free-tier; bf16 only on Ampere+ (T4/A100-no, L4/A100-80 yes)
import torch
DTYPE = torch.float16
if HAS_GPU and torch.cuda.get_device_capability(0)[0] >= 8:
    DTYPE = torch.bfloat16
print('dtype:', DTYPE, '| torch', torch.__version__)

if IN_COLAB:
    %pip -q install diffusers transformers accelerate safetensors gradio flask flask-cors imageio imageio-ffmpeg
    %pip -q install psutil
print('[+] 依赖就绪')
# Wan 2.2 TI2V-5B 需要 diffusers 主干版（PyPI 稳定版尚未包含该模型支持）。
# Anima/SDXL 用稳定版即可；要跑第 7 格视频再执行下面这行。
INSTALL_DIFFUSERS_MAIN = False
if INSTALL_DIFFUSERS_MAIN:
    %pip -q install git+https://github.com/huggingface/diffusers
    print('[+] diffusers 已换成主干版（跑 Wan 视频前需要）')
else:
    print('[i] 跑第 7 格视频前，把 INSTALL_DIFFUSERS_MAIN 改成 True 并重跑本格')


In [ ]:
#@title 2. 会话保活 + 产出落盘（防断线 / 防丢产出）
import os, time, shutil, threading

OUT_DIR = '/content/out' if IN_COLAB else './out'
os.makedirs(OUT_DIR, exist_ok=True)

# --- 保活：免费档约 90min 无交互断线。这段 JS 定期点一下 notebook。
KEEPALIVE = True
if IN_COLAB and KEEPALIVE:
    from IPython.display import display, Javascript
    display(Javascript('''
    function keepAlive() {
      const btn = document.querySelector('colab-connect-button');
      if (btn && btn.shadowRoot) {
        const c = btn.shadowRoot.querySelector('#connect');
        if (c) { c.click(); console.log('keepalive ping', new Date().toLocaleTimeString()); }
      }
    }
    setInterval(keepAlive, 60000);
    ''') )
    print('[+] keepalive 已挂载（60s 一次）')

# --- 落盘：/content 会话结束清空，产出必须搬走
DRIVE_OK = False
try:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_DIR = '/content/drive/MyDrive/ai_out'
    os.makedirs(DRIVE_DIR, exist_ok=True)
    DRIVE_OK = True
    print('[+] Drive 已挂载 ->', DRIVE_DIR)
    print('    注意：免费 Drive 仅 15GB，只存 LoRA / 成品 / 数据集')
except Exception as e:
    print('[!] Drive 未挂载（可跳过，用浏览器下载）:', str(e)[:160])

def flush_to_drive(verbose=True):
    """把 out/ 里的产出搬到 Drive，然后清空临时目录防爆盘。"""
    if not DRIVE_OK:
        if verbose: print('[!] Drive 不可用，产出留在', OUT_DIR)
        return
    tgt = os.path.join(DRIVE_DIR, time.strftime('%m%d_%H%M'))
    os.makedirs(tgt, exist_ok=True)
    n = 0
    for root, _, files in os.walk(OUT_DIR):
        for f in files:
            src = os.path.join(root, f)
            shutil.copy2(src, os.path.join(tgt, f)); n += 1
    if verbose: print(f'[+] 已搬 {n} 个文件 -> {tgt}')

print('[+] 输出目录:', OUT_DIR)

In [ ]:
#@title 3. 加载 Anima 模型（GPU 档；CPU 档自动跳过）
# Anima 官方 diffusers 权重，6.8GB，T4 装得下。
MODEL_ID = 'circlestone-labs/Anima-Base-v1.0-Diffusers'
pipe = None

if HAS_GPU:
    from diffusers import ModularPipeline
    pipe = ModularPipeline.from_pretrained(MODEL_ID, torch_dtype=DTYPE)
    pipe.load_components()          # 当前 diffusers 的正确 API
    pipe.to('cuda')
    pipe.set_progress_bar_config(disable=True)
    # 省显存：T4 只有 16GB，注意力切成内存高效实现
    try:
        pipe.enable_attention_slicing()
        print('[+] attention slicing on')
    except Exception:
        pass
    print(f'[+] Anima 已加载: {MODEL_ID}')
    print(f'    dtype={DTYPE}  VRAM={torch.cuda.memory_allocated()/1024**3:.1f}GB')
else:
    print('[!] 无 GPU，跳过模型加载。本会话只能做 ffmpeg 后处理/打标。')

In [ ]:
#@title 4. 启动 HTTP 服务 + cloudflared 公网穿透（打印 url）
import threading, uuid, os, io, json, base64, subprocess, time, re
from flask import Flask, request, jsonify, send_file
from flask_cors import CORS

app = Flask(__name__)
CORS(app)
JOBS = {}
QUEUE = []          # 批量队列，供 /batch 使用
URL = None

# ---------- 生成核心 ----------
def render(d, on_step=None):
    """单张出图。返回 PNG bytes。d 里带 prompt/负向/尺寸/步数/seed。"""
    if pipe is None:
        raise RuntimeError('no GPU / model not loaded')
    p = (d.get('prompt') or '').strip()
    if not p.startswith('masterpiece'):
        # Anima 推荐的质量前缀（Danbooru tag 风格）
        p = 'masterpiece, best quality, score_7, ' + p
    neg = d.get('negative') or 'worst quality, low quality, score_1, score_2, score_3, artist name, blurry, jpeg artifacts, chromatic aberration'
    w = int(d.get('width') or 832)
    h = int(d.get('height') or 1216)
    steps = int(d.get('steps') or 25)
    seed = int(d.get('seed') or 0)
    g = torch.Generator('cuda').manual_seed(seed)
    out = pipe(prompt=p, negative_prompt=neg, height=h, width=w,
               num_inference_steps=steps, generator=g)
    buf = io.BytesIO()
    out.images[0].save(buf, format='PNG')
    return buf.getvalue()

def work(jid, d, save=True):
    try:
        JOBS[jid]['status'] = 'running'
        t0 = time.time()
        png = render(d)
        secs = round(time.time() - t0, 1)
        upd = {'status': 'done', 'seconds': secs, 'png': png}
        if save:
            path = os.path.join(OUT_DIR, f"{jid}.png")
            open(path, 'wb').write(png)
            upd['path'] = path
        JOBS[jid].update(upd)
    except Exception as e:
        JOBS[jid].update(status='error', error=f'{type(e).__name__}: {e}'[:400])

# ---------- 路由 ----------
@app.route('/health')
def health():
    return jsonify(ok=True, gpu=HAS_GPU, gpu_name=GPU_NAME,
                   vram_gb=round(VRAM_GB, 1), queue=len(JOBS), out=OUT_DIR)

@app.route('/generate', methods=['POST'])
def generate():
    d = request.get_json(force=True)
    if not (d.get('prompt') or '').strip():
        return jsonify(error='prompt required'), 400
    jid = uuid.uuid4().hex
    JOBS[jid] = {'status': 'queued', 'created': time.time()}
    threading.Thread(target=work, args=(jid, d), daemon=True).start()
    return jsonify(id=jid)

@app.route('/batch', methods=['POST'])
def batch():
    """一次提交多条 prompt：{'prompts':[...], 'n_per_prompt':1, ...默认参数}"""
    d = request.get_json(force=True)
    prompts = d.get('prompts') or []
    if not prompts:
        return jsonify(error='prompts required'), 400
    n = int(d.get('n_per_prompt') or 1)
    ids = []
    for pr in prompts:
        for k in range(n):
            jid = uuid.uuid4().hex
            dd = dict(d); dd['prompt'] = pr
            if 'seed' in d and d['seed']:
                dd['seed'] = int(d['seed']) + k
            JOBS[jid] = {'status': 'queued', 'created': time.time()}
            threading.Thread(target=work, args=(jid, dd), daemon=True).start()
            ids.append(jid)
    return jsonify(ids=ids, count=len(ids))

@app.route('/status/<jid>')
def status(jid):
    j = JOBS.get(jid)
    if not j:
        return jsonify(error='unknown id'), 404
    return jsonify({k: v for k, v in j.items() if k != 'png'})

@app.route('/image/<jid>')
def image(jid):
    j = JOBS.get(jid)
    if not j or j.get('status') != 'done' or 'png' not in j:
        return jsonify(error='not ready'), 404
    return send_file(io.BytesIO(j['png']), mimetype='image/png')

@app.route('/flush')
def flush():
    flush_to_drive()
    return jsonify(ok=True, drive=DRIVE_OK)

threading.Thread(
    target=lambda: app.run(host='0.0.0.0', port=7860, debug=False, use_reloader=False),
    daemon=True).start()

# cloudflared quick tunnel：无需账号，地址每次重启都变
if not os.path.exists('/usr/local/bin/cloudflared'):
    %sh wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared

proc = subprocess.Popen(
    ['/usr/local/bin/cloudflared', 'tunnel', '--url', 'http://127.0.0.1:7860', '--no-autoupdate'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in proc.stdout:
    if 'trycloudflare.com' in line:
        m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', line)
        if m:
            URL = m.group(0); break

print('=' * 64)
print('PUBLIC URL:', URL)
print('=' * 64)
print('本机用法:')
print(f'  scripts/colab_gen.sh "prompt" {URL}')
print(f'  echo "{URL}" > ~/.dsh/.colab-url   # 存下来就不用每次传')

In [ ]:
#@title 5. 自测：出一张 + 验证全链路（GPU 档）
if not HAS_GPU:
    print('[!] CPU 档，跳过。请去跑第 8 格做后处理。')
else:
    import requests, time
    r = requests.post(URL + '/generate', json={
        'prompt': '1girl, solo, blue long hair, sitting on bed, looking at viewer, white t-shirt, soft morning window light, volumetric lighting, detailed skin texture',
        'width': 832, 'height': 1216, 'steps': 25, 'seed': 12345,
    })
    jid = r.json()['id']
    print('job', jid)
    while True:
        s = requests.get(URL + '/status/' + jid).json()
        print(' ', s.get('status'), s.get('seconds', ''), s.get('error', ''))
        if s.get('status') in ('done', 'error'):
            break
        time.sleep(5)
    if s.get('status') == 'done':
        from PIL import Image
        img = requests.get(URL + '/image/' + jid).content
        im = Image.open(io.BytesIO(img))
        print('  出图成功:', im.size, f'{len(img)/1024:.0f} KB, {s["seconds"]}s')
        display(im)
        print('  T4 上 25 步 832x1216 通常 15-40s；若 >120s 说明掉到 CPU 了')

In [ ]:
#@title 6. 批量出图（GPU 档；最省时间的用法）
PROMPTS = [
    # 每行一条；建议 20-40 条一批，跑 30-60 分钟
    '1girl, solo, silver hair, standing in rain, wet clothes, city night, neon reflections',
    '1girl, solo, red kimono, cherry blossoms, sunlight through petals, bokeh',
    '1girl, solo, black hair, reading book in library, warm lamp light, dust motes',
]
N_PER_PROMPT = 2
BATCH_STEPS  = 22          # 22-25 步质量/速度平衡点
W, H         = 832, 1216
SEED_BASE    = 1000

if not HAS_GPU:
    print('[!] CPU 档，跳过批量。')
else:
    import requests, time, os
    r = requests.post(URL + '/batch', json={
        'prompts': PROMPTS, 'n_per_prompt': N_PER_PROMPT,
        'width': W, 'height': H, 'steps': BATCH_STEPS, 'seed': SEED_BASE,
    })
    ids = r.json()['ids']
    print(f'[+] 已入队 {len(ids)} 张，预计 {len(ids) * 30 / 60:.0f} 分钟')
    t0 = time.time(); done = 0
    while len([i for i in ids if JOBS[i]['status'] in ('done', 'error')]) < len(ids):
        d = len([i for i in ids if JOBS[i]['status'] == 'done'])
        if d != done:
            done = d
            el = (time.time() - t0) / 60
            print(f'  {done}/{len(ids)}  用时 {el:.1f}min  剩余约 {el/max(done,1)*(len(ids)-done):.0f}min')
        time.sleep(5)
    ok = [i for i in ids if JOBS[i]['status'] == 'done']
    print(f'[+] 完成 {len(ok)}/{len(ids)}，总用时 {(time.time()-t0)/60:.1f} 分钟')
    flush_to_drive()
    print('[!] 会话结束 /content 会清空，务必确认上面已搬进 Drive')

In [ ]:
#@title 7. 视频：Wan 2.2 TI2V-5B（GPU 档；官方 720P@24fps，单卡 5 秒 <9 分钟）
# 官方 model card 用 WanPipeline（同一框架同时支持 T2V 和 I2V），
# 且明确要求 diffusers 主干版（PyPI 稳定版尚无该模型支持）。
MAKE_VIDEO = False          # 改成 True 才跑，避免误烧会话时间
SRC_IMAGE  = ''             # 图生视频填 out/ 里的静帧；留空则纯文本生视频
VID_PROMPT = 'gentle camera push in, hair swaying, soft light shift, subtle breathing motion'
VID_H, VID_W = 704, 1280    # 官方示例尺寸（720P 档）
VID_FRAMES   = 121          # 121 帧 @24fps = 5 秒；降到 49 可省一半时间
VID_STEPS    = 50           # 官方 50 步；降到 30 提速且质量尚可
VID_FPS      = 24           # TI2V-5B 原生 24fps

if not MAKE_VIDEO:
    print('[i] MAKE_VIDEO=False，跳过（改 True 再跑）')
elif not HAS_GPU:
    print('[!] 无 GPU，跳过视频。')
elif VRAM_GB < 15:
    print(f'[!] 显存 {VRAM_GB:.1f}GB 偏小；TI2V-5B 720P 建议 >=16GB（T4 需 offload）')
else:
    import time, os, subprocess
    from diffusers import WanPipeline, AutoencoderKLWan
    from diffusers.utils import export_to_video, load_image

    VID_MODEL = 'Wan-AI/Wan2.2-TI2V-5B-Diffusers'
    print(f'[*] 加载 {VID_MODEL}（首次下载约 10-20GB，视网络 5-15 分钟）')
    # VAE 固定 fp32（官方写法，精度需要），transformer 用探测到的 dtype 省显存
    vae = AutoencoderKLWan.from_pretrained(VID_MODEL, subfolder='vae', torch_dtype=torch.float32)
    vpipe = WanPipeline.from_pretrained(VID_MODEL, vae=vae, torch_dtype=DTYPE)
    # 16GB 卡必开 offload，否则 720P 会 OOM
    try:
        vpipe.enable_model_cpu_offload()
        vpipe.enable_vae_slicing()
        print('[+] cpu offload + vae slicing on')
    except Exception as e:
        vpipe.to('cuda'); print('[i] offload 不可用，直接 to(cuda):', str(e)[:80])

    # 负向用官方那串中文（对 Wan 系列是调好的）
    VID_NEG = ('色调艳丽，过曝，静态，细节模糊不清，字幕，风格，作品，画作，画面，静止，整体发灰，'
               '最差质量，低质量，JPEG压缩残留，丑陋的，残缺的，多余的手指，画得不好的手部，'
               '画得不好的脸部，畸形的，毁容的，形态畸形的肢体，手指融合，静止不动的画面，'
               '杂乱的背景，三条腿，背景人很多，倒着走')

    kw = dict(prompt=VID_PROMPT, negative_prompt=VID_NEG,
              height=VID_H, width=VID_W, num_frames=VID_FRAMES,
              guidance_scale=5.0, num_inference_steps=VID_STEPS)
    if SRC_IMAGE and os.path.exists(SRC_IMAGE):
        kw['image'] = load_image(SRC_IMAGE)   # 有静帧则走 I2V
        print('[*] 模式: 图生视频 (I2V)')
    else:
        print('[*] 模式: 文生视频 (T2V)')

    t0 = time.time()
    frames = vpipe(**kw).frames[0]
    secs = time.time() - t0
    raw = os.path.join(OUT_DIR, 'vid_raw.mp4')
    export_to_video(frames, raw, fps=VID_FPS)
    print(f'[+] {len(frames)} 帧 @{VID_FPS}fps，用时 {secs/60:.1f} 分钟')

    # 压小：Colab 临时盘 102GB，原生 mp4 很快吃光
    final = os.path.join(OUT_DIR, 'vid_final.mp4')
    subprocess.run(['ffmpeg', '-y', '-i', raw, '-c:v', 'libx264', '-crf', '23',
                    '-preset', 'fast', '-movflags', '+faststart', final], check=True)
    print(f'[+] {final}  {os.path.getsize(final)/1024**2:.1f} MB')
    flush_to_drive()


In [ ]:
#@title 8. CPU 档专用：后处理 / 打标 / 编解码（无卡时跑这格抢回时间）
# 落到 CPU 时段别去生图（单张数分钟，得不偿失），用 ffmpeg + 打标把弹药备好。
import os, glob, subprocess, shutil

MODE = 'probe'      # probe | make_frames | to_h264 | tag | dataset
SRC_DIR = OUT_DIR
print(f'MODE={MODE}  SRC_DIR={SRC_DIR}')
print('ffmpeg:', subprocess.run(['ffmpeg','-version'], capture_output=True, text=True).stdout.split('\n')[0])

def sh(cmd):
    print('$', ' '.join(cmd)[:140])
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode != 0:
        print('  [!]', (r.stderr or '')[-400:])
    return r

if MODE == 'probe':
    # 视频硬件/软件编码器可用性 —— 决定能不能快速编码
    enc = subprocess.run(['ffmpeg','-hide_banner','-encoders'], capture_output=True, text=True).stdout
    for k in ['libx264', 'libx265', 'h264_nvenc', 'av1_mediacodec']:
        print(f'  {k:18s}', 'OK' if k in enc else 'missing')
    if vm:
        print(f'  RAM 可用 {vm.available/1024**3:.1f} GB')

elif MODE == 'make_frames':
    # 视频抽帧 -> 做 LoRA 数据集 / 提取关键帧
    os.makedirs('frames', exist_ok=True)
    for v in glob.glob(os.path.join(SRC_DIR, '*.mp4')):
        sh(['ffmpeg','-y','-i',v,'-vf','fps=2','frames/%04d.png'])
    print('[+] frames/ 共', len(glob.glob('frames/*.png')), '张')

elif MODE == 'to_h264':
    # 云端出的片子可能是 H.265/ProRes，转成手机认得的 H.264
    for v in glob.glob(os.path.join(SRC_DIR, '*.mp4')):
        out = v.replace('.mp4', '_h264.mp4')
        sh(['ffmpeg','-y','-i',v,'-c:v','libx264','-crf','24','-preset','veryfast','-c:a','aac',out])

elif MODE == 'tag':
    # WD14 打标，为 LoRA 训练备数据集（CPU 也能跑，慢但够用）
    try:
        %pip -q install onnxruntime huggingface_hub pandas
        from huggingface_hub import hf_hub_download
        print('[i] WD14 打标：建议在 GPU 档跑，CPU 每小时约 300-600 张')
    except Exception as e:
        print('[!]', e)

elif MODE == 'dataset':
    # 数据集整理：统一尺寸 + 去重 + 重命名（LoRA 训练前必备）
    os.makedirs('dataset', exist_ok=True)
    from PIL import Image
    seen = set(); n = 0
    for p in sorted(glob.glob(os.path.join(SRC_DIR, '*.png'))):
        try:
            im = Image.open(p).convert('RGB')
            k = (im.size, len(im.tobytes()) % 9973)
            if k in seen:
                continue
            seen.add(k)
            side = min(im.size)
            im = im.resize((1024, int(im.height * 1024 / im.width)), Image.LANCZOS) \
                if side >= 1024 else im
            im.save(f'dataset/img_{n:05d}.png')
            n += 1
        except Exception as e:
            print('  skip', p, e)
    print(f'[+] dataset/ 共 {n} 张（去重后）')
    flush_to_drive()

## 实战注意（免费档踩坑清单）

### 会话与档位

- **时长是 3.5-6h，不是 12h**。旧版写的 12 小时是错的，别按 12h 排活；至少留 20 分钟缓冲落盘。
- **可能连到 CPU 档**。每天早晚各试一次，抢到 GPU 就跑第 6 格批量，抢不到就跑第 8 格做后处理。
- **空闲 90 分钟会断线**。第 2 格的 keepalive 已挂，但别关浏览器标签。

### 内存

- **12.6GB 是免费档标准系统内存**，不是缩水。网上说的 25.2GB 是把 15GB 显存算进去了。
- 加载 Flux / 14B 视频模型会 OOM。免费档的天花板就是 **Anima + SDXL + Wan 2.2 5B**。

### 磁盘 102GB

- **`/content` 是临时盘，会话结束即清空**。每次会话都要重下模型（带宽快，5-10 分钟可接受）。
- **别指望复用**。要复用就上 Drive，但免费 Drive 只有 15GB。
- 视频最吃盘：一条 480p/49 帧原生 mp4 约 10-40MB，批量前先看 `du -sh /content`。

### 省钱省时的三个开关

| 开关 | 效果 |
|---|---|
| `steps` 25 → 20 | 快 20%，质量损失肉眼难辨 |
| 先出图再 I2V | 比 T2V 省 3-5 倍 |
| 分辨率 `832x1216` 而非 `1024x1536` | T4 上快近一倍 |

### 相关

- 本机客户端：[scripts/colab_gen.sh](<../scripts/colab_gen.sh>)
- 免费路线全景（AI Horde / Pollinations / ZeroGPU 配额机制）：[docs/routes.md](<../docs/routes.md>)

### 视频的两条硬约束（官方 model card 确认）

- **必须装 diffusers 主干版**：`pip install git+https://github.com/huggingface/diffusers`，PyPI 稳定版还不支持 TI2V-5B。
- **官方参数是 720P / 24fps / 5 秒**（`704x1280`, `num_frames=121`），单卡消费级 GPU 也就 **9 分钟内**。别自己降到 480p/16fps，那是旧版凭感觉写的。
- 官方用 `WanPipeline`（不是 `WanImageToVideoPipeline`），同一个 pipeline 靠是否有 `image=` 参数决定走 T2V 还是 I2V。
